In [ ]:
import importlib.metadata
import os
import random
import subprocess
import sys
from pathlib import Path

def _ensure_transformers(min_ver=(5, 0, 0)):
    need_install = False
    try:
        ver_str = importlib.metadata.version("transformers").split("+")[0].split("a")[0].split("b")[0].split("rc")[0]
        parts = tuple(int(x) for x in ver_str.split(".")[:3])
        if parts < min_ver:
            need_install = True
    except Exception:
        need_install = True
    if need_install:
        print("[SETUP] Installing transformers>=5.0.0...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.0.0"], check=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    _ensure_transformers()

import numpy as np
import torch
import torchvision
import transformers

CHECKPOINT_DIR = Path("/content/checkpoints") if IN_COLAB else Path("./checkpoints")
ENSEMBLE_DIR = CHECKPOINT_DIR / "ensemble"
OUTPUT_DIR_S2 = Path("/content/stage2_results") if IN_COLAB else Path("./stage2_results")
ZIP_EXPORT_S2 = Path("/content/SE374_Stage2_Training_Results.zip") if IN_COLAB else Path("./SE374_Stage2_Training_Results.zip")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_S2.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]
CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=True)
    CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())

STAGE2_GIT_REF = os.environ.get("STAGE2_GIT_REF", "").strip()
if STAGE2_GIT_REF:
    if not (Path(CODE_ROOT) / ".git").exists():
        raise RuntimeError(f"STAGE2_GIT_REF={STAGE2_GIT_REF!r} requested, but {CODE_ROOT} is not a git repo")
    subprocess.run(["git", "-C", CODE_ROOT, "reset", "--hard", "HEAD"], capture_output=True, text=True, check=True)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", "+refs/heads/*:refs/remotes/origin/*", "--tags"], capture_output=True, text=True, check=False)
    subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--depth", "1", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=False)
    _br = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "--verify", "--quiet", f"refs/remotes/origin/{STAGE2_GIT_REF}^{{commit}}"], capture_output=True, text=True, check=False)
    if _br.returncode == 0:
        subprocess.run(["git", "-C", CODE_ROOT, "checkout", "-B", STAGE2_GIT_REF, f"refs/remotes/origin/{STAGE2_GIT_REF}"], capture_output=True, text=True, check=True)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
    else:
        if (Path(CODE_ROOT) / ".git" / "shallow").exists():
            subprocess.run(["git", "-C", CODE_ROOT, "fetch", "--unshallow", "origin"], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "checkout", "--force", STAGE2_GIT_REF], capture_output=True, text=True, check=True)
elif (Path(CODE_ROOT) / ".git").exists() and IN_COLAB:
    _hb = subprocess.run(["git", "-C", CODE_ROOT, "symbolic-ref", "--short", "-q", "HEAD"], capture_output=True, text=True, check=False)
    if _hb.returncode == 0 and _hb.stdout.strip():
        subprocess.run(["git", "-C", CODE_ROOT, "fetch", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)
        subprocess.run(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", _hb.stdout.strip()], capture_output=True, text=True, check=False)

for _k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    sys.modules.pop(_k, None)
if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

SEED = int(os.environ.get("STAGE2_SEED", "42"))
DRY_RUN = os.environ.get("STAGE2_DRY_RUN", "0").strip().lower() in ("1", "true", "yes")
NUM_WORKERS = int(os.environ.get("STAGE2_NUM_WORKERS", "4"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (
    torch.bfloat16
    if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False))
    else torch.float16
)
print(f"[READY] Code={CODE_ROOT} | Mode=visual_overlay_audit | DryRun={DRY_RUN} | Seed={SEED}", flush=True)
print(f"[READY] Compute={DEVICE} | AMP={AMP_DTYPE} | torch={torch.__version__} | tv={torchvision.__version__} | tf={transformers.__version__}", flush=True)


In [ ]:
import urllib.request
import zipfile
from torch.utils.data import DataLoader
from src.colab_pkg.downloader import GDRIVE_DATASET_ID, HF_DATASET_URL, resolve_split_dirs
from src.training.dataset_loader import BCSSStage2ValDataset, load_bcss_patient_splits

DATASET_DIR = Path("/content/BCSS-WSSS") if IN_COLAB else Path("./data/BCSS-WSSS")
DATASET_ZIP = Path("/content/BCSS-WSSS.zip") if IN_COLAB else Path("./data/BCSS-WSSS.zip")
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

_, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)
if TEST_IMG_DIR is None or TEST_MASK_DIR is None:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1_000_000:
        print(f"[DOWNLOAD] Fetching BCSS-WSSS from {HF_DATASET_URL}", flush=True)
        try:
            urllib.request.urlretrieve(HF_DATASET_URL, str(DATASET_ZIP))
        except Exception:
            subprocess.run(["pip", "install", "-q", "gdown"], check=False)
            subprocess.run(["gdown", "--id", GDRIVE_DATASET_ID, "-O", str(DATASET_ZIP)], check=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(str(DATASET_DIR))
    _, TEST_IMG_DIR, TEST_MASK_DIR = resolve_split_dirs(DATASET_DIR, split="test", min_images=1, min_patients=1)

if TEST_IMG_DIR is None or TEST_MASK_DIR is None:
    raise RuntimeError(f"[STAGE2 ERROR] Missing test split directories under {DATASET_DIR}")

splits = load_bcss_patient_splits()
test_pids = splits["test"]
assert set(test_pids).isdisjoint(set(splits["train"]) | set(splits["val"])), "Patient leakage detected!"

sample_lim = 64 if DRY_RUN else None
test_ds = BCSSStage2ValDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    patient_ids=test_pids,
    limit=sample_lim,
    strict_patient_filter=True,
    seed=SEED,
)
nw_eval = min(NUM_WORKERS, os.cpu_count() or 2)
dl_kw = {"prefetch_factor": 2, "multiprocessing_context": "spawn"} if nw_eval > 0 else {}
test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False,
    num_workers=nw_eval,
    pin_memory=(DEVICE.type == "cuda"),
    **dl_kw,
)
print(f"[DATASET] Test={len(test_ds)} patches ({len(test_pids)} disjoint patients)", flush=True)


In [ ]:
from src.colab_pkg.downloader import HF_STAGE2_URL
from src.models.wsss_inference_methods import safe_torch_load
from src.stage2_v2.ensemble import (
    DEFAULT_ARM_FAMILIES,
    SHA256_AUTHOR_STAGE2,
    SHA256_STAGE1_BEST_CLS,
    SHA256_STAGE1_RETRAINED,
    validate_track_a_arm_spec,
)
from src.stage2_v2.visual_audit import GATE7_ARM_ORDER, GATE7_ARM_SHA256
from src.training.eval_stage1_full import compute_file_sha256

def _find_ckpt(candidates, expected_sha=None, download_url=None, label="ckpt"):
    existing = [p for p in candidates if p.is_file()]
    path = None
    if expected_sha:
        for cand in existing:
            if compute_file_sha256(cand) == expected_sha:
                path = cand
                break
        if path is None and existing and not download_url:
            bad_sha = compute_file_sha256(existing[0])
            raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {label} ({existing[0]}): got {bad_sha}, expected {expected_sha}")
    elif existing:
        path = existing[0]
    if path is None and download_url:
        path = candidates[0]
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"[DOWNLOAD] Fetching {label} -> {path}", flush=True)
        urllib.request.urlretrieve(download_url, str(path))
    if path is None or not path.is_file():
        raise RuntimeError(f"[STAGE2 ERROR] Missing {label}: checked {[str(c) for c in candidates]}")
    sha = compute_file_sha256(path)
    if expected_sha and sha != expected_sha:
        raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {label} ({path}): got {sha}, expected {expected_sha}")
    print(f"[CKPT OK] {label}: {path} (sha256={sha[:12]})", flush=True)
    return path

STAGE2_TRACK = os.environ.get("STAGE2_TRACK", "track_a_clean").strip().lower()
_raw_arms = os.environ.get("STAGE2_ENSEMBLE_ARMS", "").strip()
ENSEMBLE_ARMS = [a.strip().lower() for a in _raw_arms.split(",") if a.strip()] if _raw_arms else list(GATE7_ARM_ORDER)
if len(ENSEMBLE_ARMS) != 4 or len(set(ENSEMBLE_ARMS)) != 4:
    raise ValueError(f"[STAGE2 ERROR] STAGE2_ENSEMBLE_ARMS must specify 4 distinct arm names, got: {ENSEMBLE_ARMS}")
print(f"[AUDIT CONFIG] Track={STAGE2_TRACK} | Arms={ENSEMBLE_ARMS}", flush=True)

arm_ckpt_map = {}
vps_ev_dirs = [CHECKPOINT_DIR / "vps_evidence", Path(CODE_ROOT) / "checkpoints" / "vps_evidence"]
search_dirs = [ENSEMBLE_DIR, CHECKPOINT_DIR, Path(CODE_ROOT) / "checkpoints"]
ensemble_pth_files = sorted(ENSEMBLE_DIR.glob("*.pth")) if ENSEMBLE_DIR.is_dir() else []
for arm_name in ENSEMBLE_ARMS:
    short_arm = arm_name.split("_")[0]
    exp_sha = GATE7_ARM_SHA256.get(arm_name)
    cands = []
    for d in search_dirs:
        cands.extend([d / f"{arm_name}.pth", d / f"{short_arm}.pth", d / f"{short_arm}_best.pth"])
    for ev_d in vps_ev_dirs:
        if ev_d.is_dir():
            cands.extend(sorted(ev_d.glob(f"*_g7_{short_arm}_*/stage2_retrained_best.pth")))
    existing_cands = [c for c in cands if c.is_file()]
    hit = None
    if exp_sha:
        for cand in existing_cands:
            if compute_file_sha256(cand) == exp_sha:
                hit = cand
                break
        if hit is None:
            for pth in ensemble_pth_files:
                if compute_file_sha256(pth) == exp_sha:
                    hit = pth
                    break
    if hit is None and existing_cands:
        hit = existing_cands[0]
    if hit is not None:
        arm_ckpt_map[arm_name] = hit

if len(arm_ckpt_map) < 4:
    for pth in ensemble_pth_files:
        if pth in arm_ckpt_map.values():
            continue
        try:
            meta = safe_torch_load(pth, device=torch.device("cpu"))
            preset = str(meta.get("preset", "")).strip().lower() if isinstance(meta, dict) else ""
            if preset in ENSEMBLE_ARMS and preset not in arm_ckpt_map:
                arm_ckpt_map[preset] = pth
        except Exception:
            pass

missing_arms = [a for a in ENSEMBLE_ARMS if a not in arm_ckpt_map]
if missing_arms:
    raise RuntimeError(f"[STAGE2 ERROR] Missing Stage-2 arm checkpoints for: {missing_arms} in {ENSEMBLE_DIR}")

arm_specs = []
for arm_name in ENSEMBLE_ARMS:
    ckpt_p = arm_ckpt_map[arm_name]
    validate_track_a_arm_spec({"name": arm_name, "ckpt_path": ckpt_p}, track=STAGE2_TRACK)
    sha = compute_file_sha256(ckpt_p)
    exp_sha = GATE7_ARM_SHA256.get(arm_name)
    if exp_sha and sha != exp_sha:
        raise RuntimeError(f"[STAGE2 ERROR] SHA256 mismatch for {arm_name}: got {sha}, expected {exp_sha}")
    print(f"[ARM OK] {arm_name}: {ckpt_p.name} (sha256={sha[:12]})", flush=True)
    arm_specs.append({"name": arm_name, "family": DEFAULT_ARM_FAMILIES.get(arm_name, "unknown"), "ckpt_path": ckpt_p})

s1_candidates = [
    ENSEMBLE_DIR / "stage1_best_cls.pth",
    CHECKPOINT_DIR / "stage1_best_cls.pth",
    Path(CODE_ROOT) / "checkpoints" / "stage1_best_cls.pth",
]
for ev_d in vps_ev_dirs:
    if ev_d.is_dir():
        s1_candidates.extend(sorted(ev_d.glob("*_arm_b_full/checkpoints/stage1_best_cls.pth")))
STAGE1_BEST_CLS_PATH = _find_ckpt(
    s1_candidates,
    expected_sha=SHA256_STAGE1_BEST_CLS,
    label="stage1_best_cls.pth",
)

s1_retrained_candidates = [
    ENSEMBLE_DIR / "stage1_retrained_best.pth",
    CHECKPOINT_DIR / "vps_gate4_winner" / "stage1_retrained_best.pth",
    Path(CODE_ROOT) / "checkpoints" / "vps_gate4_winner" / "stage1_retrained_best.pth",
    CHECKPOINT_DIR / "stage1_retrained_best.pth",
    Path(CODE_ROOT) / "checkpoints" / "stage1_retrained_best.pth",
]
for ev_d in vps_ev_dirs:
    if ev_d.is_dir():
        s1_retrained_candidates.extend(sorted(ev_d.glob("*_arm_b_full/checkpoints/stage1_retrained_best.pth")))
STAGE1_RETRAINED_PATH = _find_ckpt(
    s1_retrained_candidates,
    expected_sha=SHA256_STAGE1_RETRAINED,
    label="stage1_retrained_best.pth",
)

author_s2_candidates = [
    ENSEMBLE_DIR / "stage2_checkpoint_trained_on_bcss.pth",
    CHECKPOINT_DIR / "stage2_checkpoint_trained_on_bcss.pth",
    Path(CODE_ROOT) / "checkpoints" / "stage2_checkpoint_trained_on_bcss.pth",
    Path("/content/drive/MyDrive/SE374_WSSS_Checkpoints/stage2_checkpoint_trained_on_bcss.pth"),
]
AUTHOR_STAGE2_PATH = _find_ckpt(
    author_s2_candidates,
    expected_sha=SHA256_AUTHOR_STAGE2,
    download_url=HF_STAGE2_URL,
    label="stage2_checkpoint_trained_on_bcss.pth",
)


In [ ]:
from src.stage2_v2.visual_audit import run_stage2_v2_visual_audit

metrics = run_stage2_v2_visual_audit(
    arm_specs=arm_specs,
    test_loader=test_loader,
    stage1_cls_ckpt=STAGE1_BEST_CLS_PATH,
    author_stage1_ckpt=STAGE1_RETRAINED_PATH,
    author_stage2_ckpt=AUTHOR_STAGE2_PATH,
    output_dir=OUTPUT_DIR_S2,
    zip_path=ZIP_EXPORT_S2,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    enforce_strict_reconciliation=(not DRY_RUN),
    enforce_sha=True,
    enforce_palette_check=True,
    enforce_equivariance=True,
    enforce_shift_alignment=True,
    track=STAGE2_TRACK,
    dpi=300,
)
print("[VISUAL AUDIT COMPLETE] Pooled TEST mIoU (4-Arm+CGM):", metrics["pooled_test_metrics"]["ens4_cgm"]["mIoU"], flush=True)


In [ ]:
print("=== GATE-7 TRACK A VISUAL AUDIT SUMMARY ===", flush=True)
for k, v in metrics["assertions"].items():
    print(f"  Assertion {k}: passed={v.get('passed')}", flush=True)
for m_key in ("author_pure_1x", "author_step0", "arm5_cgm", "ens4_no_cgm", "ens4_cgm", "ens4_simplex_cgm"):
    if m_key in metrics["pooled_test_metrics"]:
        pm = metrics["pooled_test_metrics"][m_key]
        tag = " [secondary_exploratory]" if pm.get("secondary_exploratory") else ""
        print(f"  {m_key}: mIoU={pm['mIoU']:.4f} | mDice={pm['mDice']:.4f} | per_class={pm['per_class_iou']}{tag}", flush=True)
for m_key, bf1 in metrics["boundary_f1_at_2px"].items():
    cent = metrics["tum_centroid_offset_px"].get(m_key, {})
    print(f"  Spatial {m_key}: BoundaryF1@2px={bf1['macro_boundary_f1']:.4f} | TUM_centroid_med={cent.get('median_px', 0.0):.2f}px", flush=True)
for roi in metrics.get("whole_roi_audit", []):
    print("  Whole-ROI:", roi, flush=True)


In [ ]:
import zipfile

metrics_json = OUTPUT_DIR_S2 / "STAGE2_TRAINING_METRICS.json"
plot_png = OUTPUT_DIR_S2 / "stage2_training_loss.png"
mono_png = OUTPUT_DIR_S2 / "11_spatial_alignment_monochrome_300dpi.png"
for req_f in (metrics_json, plot_png, mono_png, ZIP_EXPORT_S2):
    if not req_f.is_file():
        raise RuntimeError(f"[STAGE2 ERROR] Missing expected deliverable: {req_f}")

with zipfile.ZipFile(ZIP_EXPORT_S2, "r") as zf:
    names = set(zf.namelist())
    for req_arc in (
        "STAGE2_TRAINING_METRICS.json",
        "stage2_training_loss.png",
        "11_spatial_alignment_monochrome_300dpi.png",
        "per_patch_visual_audit.csv",
        "per_patient_class_ious_all_models.json",
        "translation_shift_matrix.csv",
        "whole_roi_audit.json",
    ):
        if req_arc not in names:
            raise RuntimeError(f"[STAGE2 ERROR] {req_arc} missing from ZIP bundle.")

zip_mb = ZIP_EXPORT_S2.stat().st_size / (1024.0 * 1024.0)
print(f"[EXPORT OK] JSON={metrics_json} | PNG={plot_png} | ZIP={ZIP_EXPORT_S2} ({zip_mb:.2f} MB)", flush=True)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_S2))
    except Exception as exc:
        print(f"[NOTICE] Colab browser download skipped: {exc}", flush=True)
